Sascha Spors,
Professorship Signal Theory and Digital Signal Processing,
Institute of Communications Engineering (INT),
Faculty of Computer Science and Electrical Engineering (IEF),
University of Rostock,
Germany

# Data Driven Audio Signal Processing - A Tutorial with Computational Examples

Winter Semester 2023/24 (Master Course #24512)

- lecture: https://github.com/spatialaudio/data-driven-audio-signal-processing-lecture
- tutorial: https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise

Feel free to contact lecturer frank.schultz@uni-rostock.de

# Manual Model for Binary Logistic Regression with One Sigmoid Layer
- manual, i.e. from scratch implementation, of the forward/backward propagation, batch gradient descent, training handler
- we follow the derivations and coding conventions from the brilliant course https://www.coursera.org/learn/neural-networks-deep-learning, cf. especially the topics from week 2
- data synthesis and data split is done with `binary_log_reg_toy_data()` which uses scikit-learn 
- statistical measures are calculated with scikit-learn and compared to manual implementations in `evaluate()` at the end of the notebook
- the implementation uses 64-Bit double precision
- manual initialisation of model weights
- no shuffling of batch data -> vanilla batch gradient descent
- static learning rate
- hence, the training is **fully deterministic** and thus all results are precisely identical with those from other exemplary implementations
    - [binary_logistic_regression_torch.ipynb](binary_logistic_regression_torch.ipynb)
    - [binary_logistic_regression_tensorflow.ipynb](binary_logistic_regression_tensorflow.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn

from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.metrics import balanced_accuracy_score, accuracy_score

from util_binary_logistic_regression import toy_data, init_weights
from util_binary_logistic_regression import my_sigmoid, predict_class

# last manual check with '1.7.2'
sklearn.__version__

# Data

In [ ]:
X_train, Y_train, X_test, Y_test = toy_data()
N_Features = X_train.shape[1]

## Learning Parameters

In [ ]:
batch_size = X_train.shape[0] // 400
num_epochs = 10
learning_rate = 0.1

## Prepare Data

our manual implementation needs transposed data compared to typical TensorFlow, Torch implementations

In [ ]:
M_train = X_train.shape[0]
X_train_man = X_train.T
Y_train_man = Y_train[None, :]
M_train, X_train_man.shape, Y_train_man.shape

In [ ]:
# create static, unshuffled batches for training
X_batch = np.reshape(X_train_man, (2, batch_size, -1), order='F')
Y_batch = np.reshape(Y_train_man, (1, batch_size, -1), order='F')
X_batch.shape, Y_batch.shape

In [ ]:
M_test = X_test.shape[0]
X_test_man = X_test.T
Y_test_man = Y_test[None, :]
M_test, X_test_man.shape, Y_test_man.shape

## Define Loss Function

this is already hard coded

In [ ]:
def loss(y_true, y_pred):
    # vectorized loss function
    L = -(y_true * np.log(y_pred) + (1.0 - y_true) * np.log(1.0 - y_pred))
    # cost function as average of all entries in L
    J = np.mean(L, axis=1)[0]
    return J

## Init Model Parameters
to obtain reproducible results with the other implementations

In [ ]:
w1, w2, b = init_weights()
w_init = np.array([[w1], [w2]])
b_init = np.array([[b]])
print(w_init.shape)
print(b_init.shape)
print([w_init, b_init[0]])

## Train the Model

In [ ]:
# set up init model parameters
w, b = w_init, b_init

# we take all training data per epoch, we do batch gradient descent
for epoch in range(num_epochs):
    epoch_loss = 0
    for batch in range(X_batch.shape[-1]):
        # forward propagation = calc current prediction, i.e. model output
        # using the current weights and the current bias
        # forward step 1 = inner product + bias:
        Z = np.dot(w.T, X_batch[:, :, batch]) + b
        # forward step 2 = apply activation function = y hat:
        A = my_sigmoid(Z)
        # accumulate epoch loss:
        epoch_loss += loss(Y_batch[:, :, batch], A) * batch_size

        # backward propagation, start at the model output and subsequently
        # move backwards from model output to model input
        # vectorized implementation
        # step 1 dL/da = d L / d yhat:
        da = -Y_batch[:, :, batch] / A + (1 - Y_batch[:, :, batch]) / (1 - A)
        # step 2 (dL/da) * da/dz:
        dz = da * A * (1 - A)
        # step 3a dL/dw = (dL/da * da/dz) * dz/dw:
        # note that the inner product not only realizes the last multiplication
        # of the back prop chain rule, but also sums up
        # average is done by '/ batch_size' to obtain
        # the empirical risk of the current batch
        dw = np.dot(X_batch[:, :, batch], dz.T) / batch_size
        # step 3b dL/db = dL/da * da/dz * dz/db:
        # mean operation to obtain empirical risk of the current batch
        db = np.mean(dz * 1)

        # GD update rule per batch
        w = w - learning_rate * dw
        b = b - learning_rate * db

    if (epoch+1) % 1 == 0:
        A_test = my_sigmoid(np.dot(w.T, X_test_man) + b)
        loss_test = loss(Y_test_man, A_test)
        print('epoch:', epoch+1)
        print('empirical risk train:', epoch_loss / M_train,
              'empirical risk test:', loss_test)

## Check Model Parameters

In [ ]:
w, b

## Model Test

### Empirical Risk

In [ ]:
A_train = my_sigmoid(np.dot(w.T, X_train_man) + b)
loss_train = loss(Y_train_man, A_train)

A_test = my_sigmoid(np.dot(w.T, X_test_man) + b)
loss_test = loss(Y_test_man, A_test)

print('empirical risk train:', loss_train)
print('empirical risk test:', loss_test)

### Prep for Class Prediction Metrics

In [ ]:
# from here, we work with numpy rank 1 arrays, i.e. (8000,) and (2000,)
y_true_train = Y_train_man[0, :]
y_pred_train = predict_class(A_train)[0, :]
y_true_test = Y_test_man[0, :]
y_pred_test = predict_class(A_test)[0, :]

### Confusion Matrix

In [ ]:
print('confusion matrix train absolute')
print(confusion_matrix(
    y_true_train,
    y_pred_train,
    normalize=None))
print('confusion matrix train in %')
print(confusion_matrix(
    y_true_train,
    y_pred_train,
    normalize='all')*100)
print('\nconfusion matrix test absolute')
print(confusion_matrix(
    y_true_test,
    y_pred_test,
    normalize=None))
print('confusion matrix test in %')
print(confusion_matrix(
    y_true_test,
    y_pred_test,
    normalize='all')*100)

### Precision, Recall, F1Score, Support

In [ ]:
p, r, f, s = precision_recall_fscore_support(
    y_true_train, y_pred_train)
print(p, r, f, s)
p, r, f, s = precision_recall_fscore_support(
    y_true_test, y_pred_test)
print(p, r, f, s)

### Accuracy, Balanced Accuracy

We have a very balanced data set, hence both values are very similar

In [ ]:
a = accuracy_score(
    y_true_train, y_pred_train)
ba = balanced_accuracy_score(
    y_true_train, y_pred_train)
print(a, ba)

a = accuracy_score(
    y_true_test, y_pred_test)
ba = balanced_accuracy_score(
    y_true_test, y_pred_test)
print(a, ba)

## Plot Data Points and Decision Plane

In [ ]:
# get model parameters
w_tmp = np.copy(w).T

# get probabilities in the prediction plane
levels = [0.0, 0.05, 0.1, 0.37, 0.5, 0.63, 0.9, 0.95, 1]
f1, f2 = np.arange(-5, 5, 0.05), np.arange(-5, 5, 0.05)
xv, yv = np.meshgrid(f1, f2)
# the model prediction as manual one-liner, this yields a probability
prob_plane = my_sigmoid(w_tmp[0, 0] * xv + w_tmp[0, 1] * yv + b)
# hard decision boundary for classes 0,1:
# prob_plane = predict_class(prob_plane)

plt.figure(figsize=(10, 10))
plt.subplot(2, 2, 1)
plt.plot(X_train[Y_train == 1, 0],
         X_train[Y_train == 1, 1],
         "o", color='orangered', ms=1)
plt.contourf(f1, f2, prob_plane, levels=levels, cmap="RdBu_r")
plt.axis("equal")
plt.colorbar()
plt.title("training class '1' " + str(X_train.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")

plt.subplot(2, 2, 2)
plt.plot(X_train[Y_train == 0, 0],
         X_train[Y_train == 0, 1],
         "o", color='dodgerblue', ms=1)
plt.contourf(f1, f2, prob_plane, levels=levels, cmap="RdBu_r")
plt.axis("equal")
plt.colorbar()
plt.title("training class '0' " + str(X_train.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")

plt.subplot(2, 2, 3)
plt.plot(X_test[Y_test == 1, 0],
         X_test[Y_test == 1, 1],
         "o", color='orangered', ms=1)
plt.contourf(f1, f2, prob_plane, levels=levels, cmap="RdBu_r")
plt.axis("equal")
plt.colorbar()
plt.title("test class '1' " + str(X_test.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")

plt.subplot(2, 2, 4)
plt.plot(X_test[Y_test == 0, 0],
         X_test[Y_test == 0, 1],
         "o", color='dodgerblue', ms=1)
plt.contourf(f1, f2, prob_plane, levels=levels, cmap="RdBu_r")
plt.axis("equal")
plt.colorbar()
plt.title("test class '0' " + str(X_test.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")

## Manual Implementation of Metrics

In [ ]:
def evaluate(y_true, y_pred):
    y_true_tmp = np.copy(y_true)
    y_pred_tmp = np.copy(y_pred)
    y_pred_tmp = predict_class(y_pred_tmp)

    # https://www.tensorflow.org/api_docs/python/tf/math/confusion_matrix
    # The rows represent the real world (ground truth) labels
    # The matrix columns represent the prediction (inference) labels
    # real=0,pred=0    real=0,pred=1
    # real=1,pred=0    real=1,pred=1
    cm = np.zeros((2, 2), dtype=int)  # 2x2 in our example with two classes

    # correct predictions:
    # real=0,pred=0
    cm[0, 0] = int(
        np.sum(
            np.logical_and(
                np.logical_not(y_true_tmp), np.logical_not(y_pred_tmp)
            )
        )
    )
    # real=1,pred=1:
    cm[1, 1] = int(np.sum(np.logical_and(y_true_tmp, y_pred_tmp)))

    # false predictions:
    # real=0,pred=1
    cm[0, 1] = int(
        np.sum(np.logical_and(np.logical_not(y_true_tmp), y_pred_tmp))
    )

    # real=1,pred=0
    cm[1, 0] = int(
        np.sum(np.logical_and(y_true_tmp, np.logical_not(y_pred_tmp)))
    )

    M_tmp = np.sum(cm)
    print(M_tmp, y_true_tmp.shape)

    cm_in_percent = cm / M_tmp * 100

    # real=1,pred=1 related to all real=1
    recall = cm[1, 1] / np.sum(cm[1, :])  # TPR
    # real=1,pred=1 related to all pred=1
    precision = cm[1, 1] / np.sum(cm[:, 1])  # PPV

    # sum of diagonal entries (i.e. matrix trace, i.e. correct predictions)
    # related to total
    accuracy = np.sum(np.diag(cm)) / M_tmp

    # balanced F-score, F1 score
    F1_score = 2 / (1 / precision + 1 / recall)  # harmonic mean

    return cm, cm_in_percent, precision, recall, F1_score, accuracy

## Manual Check of Metrics

In [ ]:
A_train = my_sigmoid(np.dot(w.T, X_train_man) + b)
loss_test = loss(Y_train_man, A_train)
(
    cm_train,
    cm_train_percent,
    precision_train,
    recall_train,
    F1_score_train,
    accuracy_train,
) = evaluate(Y_train_man, A_train)

print('train data:')
print('loss', loss_train)
# https://www.tensorflow.org/api_docs/python/tf/math/confusion_matrix
# The rows represent the real world (ground truth) labels
# The matrix columns represent the prediction (inference) labels
print("confusion matrix:")
print("(real0,pred0)  (real0,pred1)\n(real1,pred0)  (real1,pred1)")
print("absolute\n", cm_train)
print("in %\n", cm_train_percent)
print('precision/PPV', precision_train)
print('recall/TPR', recall_train)
print('F1_score', F1_score_train * 100, '%')
print('accuray', accuracy_train)

In [ ]:
A_test = my_sigmoid(np.dot(w.T, X_test_man) + b)
loss_test = loss(Y_test_man, A_test)
(
    cm_test,
    cm_test_percent,
    precision_test,
    recall_test,
    F1_score_test,
    accuracy_test,
) = evaluate(Y_test_man, A_test)

print('test data:')
print('loss', loss_test)
print("confusion matrix:")
print("(real0,pred0)  (real0,pred1)\n(real1,pred0)  (real1,pred1)")
print("absolute\n", cm_test)
print("in %\n", cm_test_percent)
print('precision/PPV', precision_test)
print('recall/TPR', recall_test)
print('F1_score', F1_score_test * 100, '%')
print('accuray', accuracy_test)

## Copyright

- the notebooks are provided as [Open Educational Resources](https://en.wikipedia.org/wiki/Open_educational_resources)
- feel free to use the notebooks for your own purposes
- the text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/)
- the code of the IPython examples is licensed under the [MIT license](https://opensource.org/licenses/MIT)
- please attribute the work as follows: *Frank Schultz, Data Driven Audio Signal Processing - A Tutorial Featuring Computational Examples, University of Rostock* ideally with relevant file(s), github URL https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise, commit number and/or version tag, year.